# Corte 2 · Explotación analítica de la bodega Olist (Google Colab)

**Proyecto de Aula Data Warehouse y Big Data · Politécnico Grancolombiano**

Este notebook despliega la bodega del Corte 1 en Colab y la explota con cubos ROLAP:

1. Clonar el repositorio del equipo desde GitHub e instalar dependencias.
2. Configurar la **cadena de conexión estándar local** (DuckDB embebido: `duckdb:///data/olist_dw.duckdb`).
3. Ejecutar el ETL (8 reglas del Corte 1) y cargar el esquema estrella.
4. Crear los cubos ROLAP y ejecutar las operaciones OLAP: roll-up, drill-down, slice, dice y pivot.
5. Responder las 5 preguntas de negocio y versionar el resultado en GitHub.

> Ejecuta las celdas en orden con **Entorno de ejecución → Ejecutar todas**. Tarda menos de 2 minutos.

## 1. Clonar el repositorio e instalar dependencias

In [ ]:
import os, subprocess, sys

# URL del repositorio del equipo en GitHub (cámbiala por la suya)
REPO_URL = os.environ.get("REPO_URL", "https://github.com/juanjose1099322-lang/DWH.git")
BASE = "/content" if os.path.isdir("/content") else os.getcwd()
REPO_DIR = os.path.join(BASE, "olist-dw-bigdata")

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb", "pandas"], check=True)
print("Repositorio en:", os.getcwd())
print(subprocess.run(["git", "log", "--oneline", "-3"], capture_output=True, text=True).stdout)

## 2. Cadena de conexión estándar local

El proyecto usa siempre conexiones a `localhost`, así que no hay IP pública, puertos abiertos ni túneles que configurar.
En Colab el motor es **DuckDB embebido**, que guarda la bodega en un archivo dentro de la máquina virtual. La misma variable `DW_URL` apunta a PostgreSQL en Killercoda.

In [ ]:
import shutil
if not os.path.exists("config/.env"):
    shutil.copy("config/.env.example", "config/.env")
print(open("config/.env", encoding="utf-8").read())

## 3. ETL: extracción, transformación (reglas 1-8) y carga del esquema estrella

In [ ]:
r = subprocess.run([sys.executable, "etl/etl_olist.py"], capture_output=True, text=True)
print(r.stdout, r.stderr[-2000:])

In [ ]:
import duckdb, pandas as pd
pd.set_option("display.max_columns", 20)
DW_URL = "duckdb:///data/olist_dw.duckdb"          # standard local string
con = duckdb.connect(DW_URL.replace("duckdb:///", ""))
sql = lambda q: con.execute(q).df()
sql("""SELECT table_name AS tabla, estimated_size AS filas
       FROM duckdb_tables() ORDER BY table_name""")

## 4. Cubos ROLAP

Los cubos son **vistas relacionales** sobre la estrella. El motor calcula las agregaciones al vuelo con `GROUP BY CUBE`, `ROLLUP` y `GROUPING SETS`; no hay un almacenamiento multidimensional aparte (eso sería MOLAP).

In [ ]:
ddl = open("sql/02_cubos_rolap.sql", encoding="utf-8").read()
con.execute(ddl)
sql("SELECT view_name AS vista FROM duckdb_views() WHERE NOT internal ORDER BY 1")

Vista base (capa semántica): la estrella aplanada con nombres de negocio.

In [ ]:
sql('SELECT * FROM vw_cubo_base LIMIT 5')

### 4.1 Roll-up: de estado a región (jerarquía Región > Estado)

In [ ]:
sql("""SELECT region_cliente, pedidos, ticket_promedio, flete_pct_producto
       FROM cubo_region_ticket WHERE estado_cliente = 'TOTAL' ORDER BY pedidos DESC""")

### 4.2 Drill-down: de la región Nordeste a sus estados

In [ ]:
sql("""SELECT estado_cliente, pedidos, ticket_promedio, flete_pct_producto
       FROM cubo_region_ticket
       WHERE region_cliente = 'Nordeste' AND estado_cliente <> 'TOTAL'
       ORDER BY flete_pct_producto DESC""")

### 4.3 Slice: año = 2018 (top 10 categorías)

In [ ]:
sql("""SELECT categoria, ingreso_neto, pedidos, ticket_promedio
       FROM cubo_ventas_categoria_estado
       WHERE anio = '2018' AND anio_mes = 'TOTAL' AND estado_cliente = 'TOTAL' AND categoria <> 'TOTAL'
       ORDER BY ingreso_neto DESC LIMIT 10""")

### 4.4 Dice: SP, RJ y MG × 3 categorías líderes × 2018

In [ ]:
sql("""SELECT estado_cliente, categoria, ingreso_neto, pedidos
       FROM cubo_ventas_categoria_estado
       WHERE anio = '2018' AND anio_mes = 'TOTAL'
         AND estado_cliente IN ('SP','RJ','MG')
         AND categoria IN ('health_beauty','watches_gifts','bed_bath_table')
       ORDER BY estado_cliente, ingreso_neto DESC""")

### 4.5 Pivot: medio de pago en columnas, rango de cuotas en filas

In [ ]:
sql("""SELECT rango_cuotas, tipo_pago, pedidos FROM cubo_pagos_satisfaccion
       WHERE rango_valor_compra = 'TOTAL' AND rango_cuotas <> 'TOTAL' AND tipo_pago <> 'TOTAL'"""
   ).pivot_table(index="rango_cuotas", columns="tipo_pago", values="pedidos", fill_value=0).astype(int)

## 5. Preguntas de negocio del Corte 1

**P1.** Evolución mensual del ingreso neto

In [ ]:
import matplotlib.pyplot as plt
m = sql("""SELECT anio_mes, ingreso_neto FROM cubo_ventas_categoria_estado
           WHERE anio_mes BETWEEN '2017-01' AND '2018-08' AND categoria='TOTAL' AND estado_cliente='TOTAL'
           ORDER BY anio_mes""")
ax = m.plot(x="anio_mes", y="ingreso_neto", legend=False, figsize=(11, 3.5), color="#2a78d6", lw=2)
ax.set_title("Ingreso neto mensual (BRL)"); ax.set_xlabel(""); ax.grid(axis="y", alpha=.3)
plt.xticks(rotation=45); plt.tight_layout(); plt.show()

**P2 y P5.** Retraso frente a la fecha estimada y calificación

In [ ]:
r = sql("""SELECT CASE WHEN dias_retraso <= -10 THEN '1. 10+ días antes'
                       WHEN dias_retraso <= -1 THEN '2. 1-9 días antes'
                       WHEN dias_retraso = 0  THEN '3. El día estimado'
                       WHEN dias_retraso <= 3 THEN '4. 1-3 días tarde'
                       WHEN dias_retraso <= 7 THEN '5. 4-7 días tarde'
                       ELSE '6. 8+ días tarde' END AS rango_retraso,
                  ROUND(AVG(calificacion), 2) AS calificacion_prom,
                  COUNT(DISTINCT order_id)    AS pedidos
           FROM vw_cubo_base
           WHERE estado_pedido = 'delivered' AND dias_entrega IS NOT NULL AND calificacion IS NOT NULL
           GROUP BY 1 ORDER BY 1""")
r

**P5.** Rutas con mayor porcentaje de entregas tardías (cubo de logística)

In [ ]:
sql("""SELECT estado_vendedor || ' -> ' || estado_cliente AS ruta, pedidos_entregados,
              dias_entrega_prom, pct_tardios, calificacion_prom
       FROM cubo_logistica_rutas
       WHERE anio = 'TOTAL' AND estado_vendedor <> 'TOTAL' AND estado_cliente <> 'TOTAL'
         AND pedidos_entregados >= 100
       ORDER BY pct_tardios DESC LIMIT 10""")

**P4.** Medio de pago, cuotas y satisfacción

In [ ]:
sql("""SELECT rango_valor_compra, tipo_pago, pedidos, calificacion_prom
       FROM cubo_pagos_satisfaccion
       WHERE rango_cuotas = 'TOTAL' AND tipo_pago <> 'TOTAL' AND rango_valor_compra <> 'TOTAL'
       ORDER BY rango_valor_compra, pedidos DESC""")

## 6. Datos del tablero gerencial

In [ ]:
con.close()
r = subprocess.run([sys.executable, "etl/exportar_dashboard.py"], capture_output=True, text=True)
print(r.stdout, r.stderr[-1500:])
# El tablero queda en dashboard/index.html (se puede publicar con GitHub Pages)

## 7. Versionar en GitHub

En Colab, guarda un token personal de GitHub en **Secretos** (ícono de llave) con el nombre `GITHUB_TOKEN`. La celda hace commit de los cambios y los sube a la rama `main`.

In [ ]:
HACER_PUSH = False   # cambia a True para subir los cambios

subprocess.run(["git", "add", "-A"], check=True)
subprocess.run(["git", "-c", "user.name=Equipo DW", "-c", "user.email=equipo@example.com",
                "commit", "-q", "-m", "Corte 2: ejecución en Colab"], check=False)
if HACER_PUSH:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
    remoto = REPO_URL.replace("https://", f"https://{token}@")
    subprocess.run(["git", "push", remoto, "HEAD:main"], check=True)
print(subprocess.run(["git", "log", "--oneline", "-3"], capture_output=True, text=True).stdout)